### Setup environment

In [1]:
%pip install accelerate bitsandbytes torch matplotlib unsloth peft trl scikit-learn tokenizers transformers evaluate python-dotenv

Note: you may need to restart the kernel to use updated packages.


### import

In [2]:
from unsloth import FastLanguageModel
import os
import sys
import torch
import evaluate
import numpy as np
import torch.nn as nn
from trl import SFTTrainer
from tqdm.auto import tqdm
from dotenv import load_dotenv
from datasets import load_dataset
from huggingface_hub import login
from accelerate import Accelerator
from torch.utils.data import DataLoader
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, AutoConfig, TrainingArguments

/home/halogenbr/.venvs/tic-unsloth312/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Failed to load /home/halogenbr/.venvs/tic-unsloth312/lib/python3.12/site-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /home/halogenbr/.venvs/tic-unsloth312/lib/python3.12/site-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /home/halogenbr/.venvs/tic-unsloth312/lib/python3.12/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /home/halogenbr/.venvs/tic-unsloth312/lib/python3.12/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
/tmp/ipykernel_11084/1981236425.py:13: UserWarning: WARNING: Unsloth should be imported before trl, transformers, peft to ensure all optimizations are applied. Your code may run slower or encounter memory issue

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`


### Login huggingface

In [3]:
print("Logging into Hugging Face Hub...")
load_dotenv()
hf_token = os.getenv("HUGFACE_TOKEN")
if hf_token:
    login(token=hf_token)
    print("Successfully logged in to Hugging Face Hub.")
else:
    print("HUGFACE_TOKEN not found in environment variables. Please set it to log in.")

Logging into Hugging Face Hub...
Successfully logged in to Hugging Face Hub.


### Device

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

### Load dataset

In [5]:
print("Loading dataset...")
dataset = load_dataset("databricks/databricks-dolly-15k", split="train")
print("Dataset loaded successfully.")
dataset

Loading dataset...
Dataset loaded successfully.


Dataset({
    features: ['instruction', 'context', 'response', 'category'],
    num_rows: 15011
})

### Processing

In [6]:
def format_conversation(examples):
    text = []
    instruction = examples["instruction"]
    context = examples["context"]
    response = examples["response"]
    for instruction, context, response in zip(instruction, context, response):
        text.append(f"<|im_start|>user\n{instruction}\n{context}<|im_end|>\n<|im_start|>assistant\n{response}\n<|im_end|>")
    return {"text": text}
dataset = dataset.map(format_conversation, batched=True)

In [7]:
dataset[0]

{'instruction': 'When did Virgin Australia start operating?',
 'context': "Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.",
 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.',
 'category': 'closed_qa',
 'text': "<|im_start|>user\nWhen did Virgin Australia start operating?\nVirgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services

### Load Model

In [8]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name,
    max_seq_length=2048,
    # quantization_config = BitsAndBytesConfig(
    #     load_in_4bit=True,
    #     bnb_4bit_use_double_quant=True,
    #     bnb_4bit_quant_type="nf4",
    #     bnb_4bit_compute_dtype=torch.float16,
    # )
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0
)


==((====))==  Unsloth 2025.11.1: Fast Qwen2 patching. Transformers: 4.57.2.
   \\   /|    NVIDIA GeForce RTX 3060. Num GPUs = 1. Max memory: 11.999 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Not an error, but Unsloth cannot patch O projection layer with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2025.11.1 patched 24 layers with 24 QKV layers, 0 O layers and 0 MLP layers.


### Set up train and training

In [17]:
from unsloth import is_bfloat16_supported # Thêm dòng này lên trên cùng
from unsloth import is_bfloat16_supported
from trl import SFTConfig, SFTTrainer # Import thêm SFTConfig

dataset_split = dataset.train_test_split(test_size=0.1, seed=42)
train_data = dataset_split["train"]
eval_data = dataset_split["test"]

trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=eval_data,
    processing_class=tokenizer,
    args=SFTConfig(
        output_dir=model_name.split("/")[-1] + "-lora-finetuned",
        dataset_text_field="text",
        max_seq_length=2048,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        save_strategy="epoch",
        eval_strategy="epoch",
        logging_strategy="epoch",
        load_best_model_at_end=True, 
        save_total_limit=1,
        optim="adamw_torch",
        lr_scheduler_type="cosine",
        learning_rate=2e-4,
        weight_decay=0.01,
        push_to_hub=True,
        num_train_epochs=3,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
    )
)



Unsloth: Tokenizing ["text"] (num_proc=32): 100%|██████████| 1502/1502 [00:07<00:00, 209.70 examples/s]


In [18]:
print(trainer.model)
trainer.model.print_trainable_parameters()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 896, padding_idx=151654)
        (layers): ModuleList(
          (0): Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=896, out_features=896, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=896, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=896, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear(in_features=896, o

In [19]:
trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 13,509 | Num Epochs = 3 | Total steps = 2,535
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 0 of 495,114,112 (0.00% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Epoch,Training Loss,Validation Loss
1,2.920000,3.047642
2,2.923600,3.047642
3,2.920000,3.047642


Unsloth: Not an error, but Qwen2ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.


TrainOutput(global_step=2535, training_loss=2.9211975699580868, metrics={'train_runtime': 6914.607, 'train_samples_per_second': 5.861, 'train_steps_per_second': 0.367, 'total_flos': 3.477044569358131e+16, 'train_loss': 2.9211975699580868, 'epoch': 3.0})

In [20]:
trainer.push_to_hub(model_name.split("/")[-1] + "-lora-finetuned")

Upload 0 LFS files: 0it [00:00, ?it/s]


CommitInfo(commit_url='https://huggingface.co/HalogenFlo/Qwen2.5-0.5B-Instruct-lora-finetuned/commit/7e17ce7660bde570f8e25f213240b0833cf387fc', commit_message='Qwen2.5-0.5B-Instruct-lora-finetuned', commit_description='', oid='7e17ce7660bde570f8e25f213240b0833cf387fc', pr_url=None, repo_url=RepoUrl('https://huggingface.co/HalogenFlo/Qwen2.5-0.5B-Instruct-lora-finetuned', endpoint='https://huggingface.co', repo_type='model', repo_id='HalogenFlo/Qwen2.5-0.5B-Instruct-lora-finetuned'), pr_revision=None, pr_num=None)

### Test

In [1]:
import torch
from unsloth import FastLanguageModel
from peft import PeftModel
from transformers import AutoTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-0.5B-Instruct",
    max_seq_length=2048,
    dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    load_in_4bit=False,
)

model = PeftModel.from_pretrained(model, "Qwen2.5-0.5B-Instruct-lora-finetuned")

model = model.to(torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16)

FastLanguageModel.for_inference(model)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/halogenbr/.venvs/tic-unsloth312/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[torchao|WARNING]Failed to load /home/halogenbr/.venvs/tic-unsloth312/lib/python3.12/site-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /home/halogenbr/.venvs/tic-unsloth312/lib/python3.12/site-packages/torchao/_C_cutlass_90a.abi3.so
[torchao|WARNING]Failed to load /home/halogenbr/.venvs/tic-unsloth312/lib/python3.12/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /home/halogenbr/.venvs/tic-unsloth312/lib/python3.12/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so


🦥 Unsloth Zoo will now patch everything to make training faster!


Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`


==((====))==  Unsloth 2025.11.1: Fast Qwen2 patching. Transformers: 4.57.2.
   \\   /|    NVIDIA GeForce RTX 3060. Num GPUs = 1. Max memory: 11.999 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 896, padding_idx=151654)
        (layers): ModuleList(
          (0-23): 24 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=896, out_features=896, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=896, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=896, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear(in_feature

In [2]:
def generate_response(prompt):
    formatted_prompt = f"<|im_start|>user\n{prompt}\n<|im_end|>\n<|im_start|>assistant\n"
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_length=2048, 
            do_sample=False, 
            repetition_penalty=1.15, 
            top_p=0.9,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id
        )
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
    return response


test_prompt = "What is the capital of France?"
response = generate_response(test_prompt)
print("Prompt:", test_prompt)
print("Response:", response)

Prompt: What is the capital of France?
Response: The capital of France is Paris.
